# One-pixel attack

Goal:
Change one single pixel in a CIFAR-10 image so that ResNet-18 misclassifies it, using Differential Evolution (DE) — no gradients, pure black-box optimization.

## Background
We don't know the model internal(i.e. black box attack)
Differential Evolution (DE)
We query the model and observe the outputs
5 parameters: (x_position, y_position, R, G, B)
Mutation Strategy: This is how you we create "children" pixels from "parent" pixels. By taking three random candidates (a,b,c) and combine them: mutant=a+F⋅(b−c), where F is a scaling factor.
In the report they use a scaling factor of 0.5 so we will use the same. 

In [3]:
from keras.datasets import cifar10
import numpy as np
import random
import matplotlib.pyplot as plt
import torch
import torchvision
import torchvision.transforms as transforms

In [10]:
# We use a transform to resize the images from the CIFAR-10 dataset as standard ResNet expects 224x224

transform = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(), # 1. Convert the PIL Image to a Tensor first
    transforms.Normalize(  # 2. Now you can do math on the Tensor
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
]) # Documentation found: https://docs.pytorch.org/vision/main/models/generated/torchvision.models.resnet18.html

# Download the CIFAR-10 Dataset
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=1, shuffle=True)

# Load a Pre-trained ResNet-18
model = torchvision.models.resnet18(weights='IMAGENET1K_V1') #https://docs.pytorch.org/vision/main/models/generated/torchvision.models.resnet18.html
model.eval() # Set to evaluation mode (no training needed)

print("Model and Data are ready!")

Model and Data are ready!


In [11]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# 1. Grab one batch from your testloader
images, labels = next(iter(testloader))
image, label = images[0], labels[0]

# 2. Make the prediction
with torch.no_grad():
    # Model expects a batch, so we add a dimension with .unsqueeze(0)
    output = model(image.unsqueeze(0))
    
    # Convert output to probabilities
    probabilities = F.softmax(output, dim=1)
    
    # Get the top prediction and its confidence
    conf, predicted_idx = torch.max(probabilities, 1)

# 3. Print the results
print(f"Original CIFAR Label ID: {label}")
print(f"Model's Top ImageNet ID: {predicted_idx.item()}")
print(f"Confidence: {conf.item() * 100:.2f}%")

Original CIFAR Label ID: 4
Model's Top ImageNet ID: 101
Confidence: 36.57%


## ResNet
Resnet is trained on images that is 224x224 so we will modify the final layer for 10 classes as this is the strategy to be able to predict images of 32x32 size. 

## Initialize population